# BRIDGET: CDC DIABETES

From: https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset

This version was chosen from Kaggle because of its size (it's a perfect 50/50 split and data must be scaled only)

## Dataset Preprocessing


### Libraries, Retrieving data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
import yaml
import joblib
import random
import time
import functools
import pickle
import re
import orjson
import alibi
import ignite
import copy

from IPython import display
from itertools import combinations, product
from tqdm import tqdm
from matplotlib import pyplot as plt
from collections import Counter

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optimizer


from river import rules, tree, datasets, drift, metrics, evaluate
from river import imblearn
from river import preprocessing
from river import optim
from river import metrics
from river import feature_extraction, feature_selection
from river import ensemble, linear_model, forest, compose

from torchsummary import summary
from torch.utils.data import TensorDataset, DataLoader

from ignite.metrics import Accuracy, Loss
from ignite.engine import Engine, Events, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import EarlyStopping, ModelCheckpoint
from ignite.contrib.handlers import global_step_from_engine
from sklearn.preprocessing import MinMaxScaler
from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

from alibi.explainers.cfproto import CounterFactualProto

from bridget_utils import *
from orchestrator import *
from master_config import *
from classes import BetaUser, DeferralNet, PyTorchWrapper, RiverModelWrapper
from bridget_main import BRIDGET, HiC, MiC


c:\Users\virgm\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


26-Apr-29 17:40:08 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [3]:
set_all_seeds(42)

In [4]:
ds_name= 'cdc'

In [5]:
data= pd.read_csv(fr"./datasets/{ds_name}_binary.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70692 entries, 0 to 70691
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Diabetes_binary       70692 non-null  float64
 1   HighBP                70692 non-null  float64
 2   HighChol              70692 non-null  float64
 3   CholCheck             70692 non-null  float64
 4   BMI                   70692 non-null  float64
 5   Smoker                70692 non-null  float64
 6   Stroke                70692 non-null  float64
 7   HeartDiseaseorAttack  70692 non-null  float64
 8   PhysActivity          70692 non-null  float64
 9   Fruits                70692 non-null  float64
 10  Veggies               70692 non-null  float64
 11  HvyAlcoholConsump     70692 non-null  float64
 12  AnyHealthcare         70692 non-null  float64
 13  NoDocbcCost           70692 non-null  float64
 14  GenHlth               70692 non-null  float64
 15  MentHlth           

In [6]:
for c in data:
    print(data[c].value_counts().sum)

<bound method Series.sum of Diabetes_binary
0.0    35346
1.0    35346
Name: count, dtype: int64>
<bound method Series.sum of HighBP
1.0    39832
0.0    30860
Name: count, dtype: int64>
<bound method Series.sum of HighChol
1.0    37163
0.0    33529
Name: count, dtype: int64>
<bound method Series.sum of CholCheck
1.0    68943
0.0     1749
Name: count, dtype: int64>
<bound method Series.sum of BMI
27.0    6327
26.0    4975
28.0    4583
24.0    4392
30.0    4344
        ... 
85.0       1
83.0       1
80.0       1
78.0       1
74.0       1
Name: count, Length: 80, dtype: int64>
<bound method Series.sum of Smoker
0.0    37094
1.0    33598
Name: count, dtype: int64>
<bound method Series.sum of Stroke
0.0    66297
1.0     4395
Name: count, dtype: int64>
<bound method Series.sum of HeartDiseaseorAttack
0.0    60243
1.0    10449
Name: count, dtype: int64>
<bound method Series.sum of PhysActivity
1.0    49699
0.0    20993
Name: count, dtype: int64>
<bound method Series.sum of Fruits
1.0    43249


### Preprocessing Pipeline
1. Drop duplicates

2. Scale every non binary feat with Min Max Scaler (for the NN)
scale (Education, Income, Age PhysHlth, MentHlth, GenHlth, BMI)
note that Mental Health and PhysHlth represents days 0 to 30

In [7]:
data= clean_compas(data, drop_duplicates=True)

In [8]:
data.info()
data.head(n=5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69057 entries, 0 to 69056
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Diabetes_binary       69057 non-null  float64
 1   HighBP                69057 non-null  float64
 2   HighChol              69057 non-null  float64
 3   CholCheck             69057 non-null  float64
 4   BMI                   69057 non-null  float64
 5   Smoker                69057 non-null  float64
 6   Stroke                69057 non-null  float64
 7   HeartDiseaseorAttack  69057 non-null  float64
 8   PhysActivity          69057 non-null  float64
 9   Fruits                69057 non-null  float64
 10  Veggies               69057 non-null  float64
 11  HvyAlcoholConsump     69057 non-null  float64
 12  AnyHealthcare         69057 non-null  float64
 13  NoDocbcCost           69057 non-null  float64
 14  GenHlth               69057 non-null  float64
 15  MentHlth           

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,0.0,0.0,15.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,3.0,0.0,29.0,0.0,0.0,7.0,5.0,2.0
1,0.0,0.0,0.0,0.0,16.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,5.0,5.0
2,0.0,0.0,0.0,0.0,16.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,2.0,0.0,0.0,0.0,0.0,6.0,5.0,6.0
3,0.0,0.0,0.0,0.0,16.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,2.0,0.0,4.0,0.0,0.0,3.0,6.0,8.0
4,0.0,0.0,0.0,0.0,17.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,3.0,0.0,0.0,1.0,0.0,13.0,4.0,2.0


### Splitting and Transforming data

1. Apply stratified sampling

2. Get pre-training/HiC/calibration/Mic data

3. Apply scaler

4. Get X, y

In [10]:
# Qui definiamo i vari split dei flussi 
set_all_seeds(42)
data = data.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle iniziale

class_0 = data[data['Diabetes_binary'] == 0]
class_1= data[data['Diabetes_binary'] == 1]

#  split ufficiale

splits= {
    'calibration': (0.6, 0.8),
    'mic': (0.8, 1.0),
    'avv_train': (0.0, 0.07),
    'avv_test': (0.07, 0.1),
    'hic_train': (0.1, 0.5),
    'hic_test': (0.5, 0.6)
}

dfs= {}

for name, (start, end) in splits.items():
    dfs[name]= stratif(start, end, class_0, class_1)



In [11]:
for name, df in dfs.items():
    print(f"{name} length: {len(df)}")


calibration length: 13811
mic length: 13812
avv_train length: 4833
avv_test length: 2072
hic_train length: 27623
hic_test length: 6906


In [12]:
target= 'Diabetes_binary'
numericals= ['Education', 'Income', 'Age', 'PhysHlth', 'MentHlth', 'GenHlth','BMI']
categoricals= [c for c in data.columns if c not in numericals and c != target]

prepr_transf = (
    (compose.Select(*numericals) | preprocessing.MinMaxScaler()) +
    compose.Select(*categoricals)
)

In [13]:
## ora divisione in x e y

set_all_seeds(42)

# avviamento 
X_avv_train, y_avv_train = x_y_split(dfs['avv_train'], target)
X_avv_test, y_avv_test = x_y_split(dfs['avv_test'], target)


# hic
X_hic_train, y_hic_train = x_y_split(dfs['hic_train'], target)
X_hic_test, y_hic_test = x_y_split(dfs['hic_test'], target)

# validation
X_val, y_val = x_y_split(dfs['calibration'], target)

# mic
X_mic, y_mic = x_y_split(dfs['mic'], target)



## Calibration Phase: Experts and Incremental Model Selection

### Calibrating Incremental Model

The incremental model to be chosen for Bridget is trained on the X_avv, y_avv portion of the dataset,then evaluated on the X_avv_test and y_avv_test

The calibration phase starts by assessing the results of the learning for several configurations:

    - HoeffdingTreeClassifier

    - ExtremelyFastDecisionTreeClassifier

    - AdaBoostClassifier            (base= SGTClassifier)

    - AdwinBaggingClassifier        (base= SGTClassifier)

    - SRPClassifier                 (base= SGTClassifier)

    - AdaptiveRandomForestClassifier


The metrics observed are the Accuracy, the F1Score and the Counters for the classes

In [14]:
# since all River models work with dicts, lets first transform the dfs to dict
# then the models are instantiated and trained by the HiC.train function
# the HiC object is initialized by passing a random user model, its not relevant since it won't interact with the IL anyways

set_all_seeds(42)

X_avv_dict= X_avv_train.to_dict(orient='records')
X_avv_dict_test= X_avv_test.to_dict(orient='records')

df_batch_1 = (dfs['hic_train']).reset_index(drop=True)
mic_data= dfs['mic'].reset_index(drop=True)
df_avv= pd.concat([dfs['avv_train'], dfs['avv_test']]).reset_index(drop=True)

test_batch_1= X_hic_test.copy()
test_batch_1[target]= y_hic_test

expert= 'accurate_trusting'

set_all_seeds(42)

base = tree.HoeffdingTreeClassifier(grace_period=100)

htree= tree.HoeffdingAdaptiveTreeClassifier(grace_period= 100, seed= 42)
efdt= tree.ExtremelyFastDecisionTreeClassifier(grace_period=100)
ada= ensemble.AdaBoostClassifier(model= base, n_models= 15, seed= 42)  
adwin= ensemble.ADWINBaggingClassifier(model= base, n_models= 15, seed= 42)
srp= ensemble.SRPClassifier(model= base, n_models=15, seed= 42)
arf= forest.ARFClassifier(n_models= 15, grace_period= 100, max_features='sqrt', seed=42)

models= {
    'HoeffdingATC': htree, 
    "EFDT": efdt, 
    "AdaBoostCl":ada, 
    "ADWINBAGGING": adwin, 
    "SRPCL": srp, 
    "ARF":arf   
    }

prepr_dir= DATASETS[ds_name]['base_obj_paths']['preprocessors']
os.makedirs(prepr_dir, exist_ok=True)

model_dir= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
os.makedirs(model_dir, exist_ok=True)

for model_name, model_obj in models.items():
    
    bridget_inst= HiC(RULE= FRANK_RULES['RULE'],
                PAST= FRANK_RULES['PAST'], 
                SKEPT=FRANK_RULES['SKEPT'], 
                GROUP= FRANK_RULES['GROUP'], 
                EVA=FRANK_RULES['EVA'], 
                n_bins=FRANK_RULES['N_BINS'], 
                n_var=FRANK_RULES['N_VAR'], 
                maxc=FRANK_RULES['MAXC'], 
                rule_att=DATASETS[ds_name]['rule_att'], 
                rule_value=DATASETS[ds_name]['rule_value'], 
                hic_model_name='placeholder', 
                hic_model=model_obj,
                start_performance= DATASETS[ds_name]['start_performance'],
                allocated_budget= DATASETS[ds_name]['allocated_budget'][0],
                skepticism_threshold= DATASETS[ds_name]['skepticism_threshold'],
                performance_delta= DATASETS[ds_name]['performance_delta'],
                dataset_name= ds_name,
                user_name= 'placeholder',
                batch1=df_batch_1, batch3=mic_data, batch1_test=test_batch_1, 
                target=target, 
                user_model='placeholder', 
                protected=DATASETS[ds_name]['protected'], cats=categoricals, num=numericals,
                preprocessor=prepr_transf,
                training_iter= 0 
                )
    
    bridget_inst.train(X_avv_dict, y_avv_train, X_avv_dict_test, y_avv_test)

    base_preprocessor = bridget_inst.preprocessor
    base_model = bridget_inst.hic_model
    

    prepr_filename = f"{model_name}_preprocessor.pkl"
    model_filename = f"{model_name}_model.pkl"

    joblib.dump(base_preprocessor, os.path.join(prepr_dir, prepr_filename))
    joblib.dump(base_model, os.path.join(model_dir, model_filename))


trained_arf= arf

Accuracy: 67.18%
F1: 64.47%
Distribution of predictions: Counter({0: 1211, 1: 861})
HoeffdingAdaptiveTreeClassifier trained
Accuracy: 68.82%
F1: 72.25%
Distribution of predictions: Counter({1: 1275, 0: 797})
ExtremelyFastDecisionTreeClassifier trained
Accuracy: 68.44%
F1: 69.50%
Distribution of predictions: Counter({1: 1091, 0: 981})
AdaBoostClassifier(HoeffdingTreeClassifier) trained
Accuracy: 71.14%
F1: 71.92%
Distribution of predictions: Counter({1: 1077, 0: 995})
ADWINBaggingClassifier(HoeffdingTreeClassifier) trained
Accuracy: 73.02%
F1: 75.10%
Distribution of predictions: Counter({1: 1192, 0: 880})
SRPClassifier(HoeffdingTreeClassifier) trained
Accuracy: 73.02%
F1: 75.12%
Distribution of predictions: Counter({1: 1194, 0: 878})
ARFClassifier trained


### Calibrating Experts


In [15]:
with open(fr"./experts_{ds_name}.yaml", "r") as f:
    config= yaml.safe_load(f)


params_dict= config['experts']['groups']['w_dict']

In [16]:
set_all_seeds(42)
X_exp= X_hic_train.to_dict(orient='records')

X_exp_scaled= []

for x in X_exp:
    X_exp_scaled.append(prepr_transf.transform_one(x))

X_exp_final = pd.DataFrame(X_exp_scaled)

In [17]:
set_all_seeds(42)
experts_obj= {}

expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

for name in expert_names:
    expert_type= config['experts']['groups'][name]

    experts_obj[name]= BetaUser(
        belief_level= expert_type['belief_value'],
        rethink_level= 0.8, # as suggested by the og FRANK implementation
        fairness= True,
        fpr= expert_type['target_FPR'],
        fnr= expert_type['target_FNR'],
        alpha= 0.9,
        features_dict= params_dict,
        seed= expert_type['group_seed']
        )
    res = experts_obj[name].fit(X_exp_final, y_hic_train, tol= 0.001)

    save_dir= fr"./trained_experts/{ds_name}"
    os.makedirs(save_dir, exist_ok= True)
    file_path = os.path.join(save_dir, f"{name}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(experts_obj[name], f)
    print(f"Expert saved in: {file_path}")

    
    print(f"{'='*30}")
    print(f" EXPERT CALIBRATION REPORT ")
    print(f"{'='*30}")

    print(f"\n[EXPERT: {name}]")
    print(f"\n[FALSE POSITIVE RATE]")
    print(f"  - Iters:      {res['fpr iters number']}")
    print(f"  - Beta:       {res['calibrated_fpr_beta']:.4f}")
    print(f"  - Target:     {res['target_fpr']}")
    print(f"  - Achieved:   {res['achieved_fpr']:.4f}")

    print(f"\n[FALSE NEGATIVE RATE]")
    print(f"  - Iters:      {res['fnr iters number']}")
    print(f"  - Beta:       {res['calibrated_fnr_beta']:.4f}")
    print(f"  - Target:     {res['target_fnr']}")
    print(f"  - Achieved:   {res['achieved_fnr']:.4f}")
    


Expert saved in: ./trained_experts/cdc\accurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      11
  - Beta:       -3.2227
  - Target:     0.1
  - Achieved:   0.0990

[FALSE NEGATIVE RATE]
  - Iters:      13
  - Beta:       -0.9033
  - Target:     0.1
  - Achieved:   0.0992
Expert saved in: ./trained_experts/cdc\accurate_not_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_not_trusting]

[FALSE POSITIVE RATE]
  - Iters:      6
  - Beta:       -3.1250
  - Target:     0.1
  - Achieved:   0.1009

[FALSE NEGATIVE RATE]
  - Iters:      10
  - Beta:       -0.9766
  - Target:     0.1
  - Achieved:   0.1000
Expert saved in: ./trained_experts/cdc\inaccurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: inaccurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      14
  - Beta:       -1.8677
  - Target:     0.3
  - Achieved:   0.2998

[FALSE NEGATIVE RATE]
  - Iters:      14
  - Beta:       0.5737
  - Target:     0.3
  - A

## BRIDGET decision making



In [ ]:
# base params for all users
initial_ordering= [c for c in data if c != target]

# retrieve preprocessor and incremental learner
rules= FRANK_RULES


base_params= {
    "cats": categoricals, #lst
    "num": numericals, #lss
    "warm_up_set": df_avv.copy(),
    "batch1": df_batch_1.copy(),
    "batch1_test": test_batch_1.copy(),
    "batch3":dfs['mic'].copy(),
    "validation_set": dfs['calibration'].copy(), #obtained by hic
    "feature_order": initial_ordering.copy(), #without the label ?
    "incremental_learner_name":"ARF",
    "n_cols": len(initial_ordering), #obtained by hic
    "mic_model_name": "Def_Net",
    "strat_1_name": "Confidence",
    "strat_2_name": "Mao",
    
}


learner_path= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
path= os.path.join(learner_path, f"{base_params["incremental_learner_name"]}_model.pkl")
trained_learner= joblib.load(path)

prepr_path= DATASETS[ds_name]['base_obj_paths']['preprocessors']
path= os.path.join(prepr_path, f"{base_params["incremental_learner_name"]}_preprocessor.pkl")
base_prepr= joblib.load(path)

base_objects = {
      "preprocessor": copy.deepcopy(base_prepr), #or take from path
      "incremental_learner":copy.deepcopy(trained_learner), # or take from path
      "scaler": None
} 

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device= torch.device("cpu")
#add these once you get to mic phase: "run_confidence": True, "run_mao": True

### Expert: Accurate, Trusting 

#### HIC

In [ ]:
#setting up fixed params

current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [ ]:
run_hic(ds_name, params, objects)

#### CALIB

In [ ]:
#chosen nets: "large", 32/16

net_layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']

# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{net_layers[0]}_{net_layers[1]}"
                          )

    pattern = f"{net_layers[0]}_{net_layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [ ]:
current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2
                            )
    
  




### Expert: Inaccurate, Trusting 

#### HIC

In [ ]:
#setting up fixed params

current_user_name= "inaccurate_trusting"
user_suffix= 'inacc_t'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [ ]:
run_hic(ds_name, params, objects)
       

#### Calib

In [ ]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']
# retrieving df and best net path

for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [ ]:
current_user_name= "inaccurate_trusting"
user_suffix= 'inacc_t'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2)
    
  

### Expert: Accurate, Not Trusting 

#### HIC

In [ ]:
#setting up fixed params

current_user_name= "accurate_not_trusting"
user_suffix= 'acc_nt'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [ ]:
run_hic(ds_name, params, objects)

#### CALIB

In [ ]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']
# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [ ]:
current_user_name= "accurate_not_trusting"
user_suffix= 'acc_nt'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2)
    
  

### Expert: Inaccurate, Not Trusting 

#### HIC

In [ ]:
#setting up fixed params

current_user_name= "inaccurate_not_trusting"
user_suffix= 'inacc_nt'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [ ]:
run_hic(ds_name, params, objects)

#### CALIB

In [ ]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']

# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [ ]:
current_user_name= "inaccurate_not_trusting"
user_suffix= 'inacc_nt'

exp_path= fr"./trained_experts/{ds_name}/{current_user_name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2)
    
  


